# Multi-Model Prompt Comparison: OpenAI vs Anthropic vs Gemini vs HuggingFace

This notebook runs the **same prompt tests** from `test_prompts.ipynb` across multiple LLM providers to compare:
- **Quality** of outputs (correctness, completeness, JSON compliance)
- **Latency** (response time)
- **Token usage** and estimated cost
- **Consistency** across providers

## Models Compared

| Tier | OpenAI | Anthropic | Google Gemini | HuggingFace (Open-Source) |
|------|--------|-----------|---------------|---------------------------|
| Fast/Cheap | gpt-4.1-mini | claude-3-5-haiku | gemini-2.0-flash | Llama 4 Scout via HF Router |
| Top | gpt-4.1 | claude-sonnet-4-5 | gemini-2.5-pro | Qwen 2.5-72B via HF Router |

## Tests Covered
1. Intent Classification
2. Project Description
3. Variable Relevance Grading
4. Cross-Analysis Pattern Extraction (V1, V2, V3)
5. Expert Summary
6. Transversal Analysis
7. Full Pipeline (end-to-end)

## HuggingFace as a Testing Platform

HuggingFace Inference Providers offer an **OpenAI-compatible API** to access 200+ open-source models.
- Free tier: $0.10/month credit (enough for ~10-20 test runs)
- PRO ($9/month): $2.00 credit + pay-as-you-go
- Same `OpenAI` client, just change `base_url` to `https://router.huggingface.co/v1`
- Append `:cheapest` or `:fastest` to model names for automatic provider routing

In [ ]:
# === SETUP ===
import sys, os, json, time, re
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML
from collections import defaultdict

# Project root
PROJECT_ROOT = Path('/workspaces/navegador')
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

# Check API keys
OPENAI_OK = bool(os.getenv('OPENAI_API_KEY'))
ANTHROPIC_OK = bool(os.getenv('ANTHROPIC_API_KEY'))
GEMINI_OK = bool(os.getenv('GOOGLE_API_KEY'))
HF_OK = bool(os.getenv('HF_TOKEN'))

print(f'OpenAI API key:    {"OK" if OPENAI_OK else "MISSING"}')
print(f'Anthropic API key: {"OK" if ANTHROPIC_OK else "MISSING"}')
print(f'Gemini API key:    {"OK" if GEMINI_OK else "MISSING"}')
print(f'HuggingFace token: {"OK" if HF_OK else "MISSING"}')

ACTIVE_PROVIDERS = []
if OPENAI_OK: ACTIVE_PROVIDERS.append('openai')
if ANTHROPIC_OK: ACTIVE_PROVIDERS.append('anthropic')
if GEMINI_OK: ACTIVE_PROVIDERS.append('gemini')
if HF_OK: ACTIVE_PROVIDERS.append('huggingface')
print(f'\nActive providers: {ACTIVE_PROVIDERS}')

In [ ]:
# === INITIALIZE LLM CLIENTS ===

# OpenAI
if OPENAI_OK:
    from openai import OpenAI
    openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# Anthropic
if ANTHROPIC_OK:
    import anthropic
    anthropic_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

# Gemini
if GEMINI_OK:
    import google.generativeai as genai
    genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))

# HuggingFace (uses OpenAI-compatible API via the HF Router)
if HF_OK:
    from openai import OpenAI as HFOpenAI
    hf_client = HFOpenAI(
        api_key=os.getenv('HF_TOKEN'),
        base_url='https://router.huggingface.co/v1'
    )

print('Clients initialized.')

In [ ]:
# === MODEL CONFIGURATIONS ===
# Each provider gets a "fast" tier model for fair comparison.
# These are the models used for the main tests.

MODEL_CONFIG = {
    'openai': {
        'fast': 'gpt-4.1-mini-2025-04-14',
        'top': 'gpt-4.1-2025-04-14',
        # Pricing per 1M tokens (USD) - approximate
        'cost_per_1m_input': 0.40,   # gpt-4.1-mini input
        'cost_per_1m_output': 1.60,  # gpt-4.1-mini output
    },
    'anthropic': {
        'fast': 'claude-3-5-haiku-20241022',
        'top': 'claude-sonnet-4-5-20250514',
        'cost_per_1m_input': 0.80,   # haiku input
        'cost_per_1m_output': 4.00,  # haiku output
    },
    'gemini': {
        'fast': 'gemini-2.0-flash',
        'top': 'gemini-2.5-pro-preview-05-06',
        'cost_per_1m_input': 0.10,   # flash input
        'cost_per_1m_output': 0.40,  # flash output
    },
    'huggingface': {
        # Llama 4 Scout via HuggingFace Inference Providers
        # Append :cheapest to let HF route to cheapest available provider
        'fast': 'meta-llama/Llama-4-Scout-17B-16E-Instruct',
        'top': 'Qwen/Qwen2.5-72B-Instruct',
        # HF Inference Providers pricing varies by backend provider;
        # these are approximate rates via the cheapest provider
        'cost_per_1m_input': 0.18,   # Llama 4 Scout typical
        'cost_per_1m_output': 0.18,  # Llama 4 Scout typical
    },
}

print('Model configurations:')
for provider, cfg in MODEL_CONFIG.items():
    status = 'ACTIVE' if provider in ACTIVE_PROVIDERS else 'SKIPPED'
    print(f'  {provider}: {cfg["fast"]} [{status}]')

In [ ]:
# === UNIFIED LLM CALL FUNCTION ===

def call_llm(prompt, provider='openai', tier='fast', temperature=0.5, system_prompt=None):
    """
    Unified LLM call across providers with timing and token tracking.
    
    Returns: (content, metadata_dict)
    """
    model = MODEL_CONFIG[provider][tier]
    start = time.time()
    
    if provider == 'openai':
        messages = []
        if system_prompt:
            messages.append({'role': 'system', 'content': system_prompt})
        messages.append({'role': 'user', 'content': prompt})
        
        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature
        )
        elapsed = time.time() - start
        content = response.choices[0].message.content
        usage = response.usage
        input_tokens = usage.prompt_tokens
        output_tokens = usage.completion_tokens
        total_tokens = usage.total_tokens
    
    elif provider == 'anthropic':
        kwargs = {
            'model': model,
            'max_tokens': 4096,
            'temperature': temperature,
            'messages': [{'role': 'user', 'content': prompt}]
        }
        if system_prompt:
            kwargs['system'] = system_prompt
        
        response = anthropic_client.messages.create(**kwargs)
        elapsed = time.time() - start
        content = response.content[0].text
        input_tokens = response.usage.input_tokens
        output_tokens = response.usage.output_tokens
        total_tokens = input_tokens + output_tokens
    
    elif provider == 'gemini':
        full_prompt = prompt
        if system_prompt:
            full_prompt = f"{system_prompt}\n\n{prompt}"
        
        gmodel = genai.GenerativeModel(
            model,
            generation_config={'temperature': temperature}
        )
        response = gmodel.generate_content(full_prompt)
        elapsed = time.time() - start
        content = response.text
        # Gemini usage metadata
        usage_meta = response.usage_metadata
        input_tokens = getattr(usage_meta, 'prompt_token_count', 0)
        output_tokens = getattr(usage_meta, 'candidates_token_count', 0)
        total_tokens = getattr(usage_meta, 'total_token_count', input_tokens + output_tokens)
    
    elif provider == 'huggingface':
        # HuggingFace Inference Providers use OpenAI-compatible API
        messages = []
        if system_prompt:
            messages.append({'role': 'system', 'content': system_prompt})
        messages.append({'role': 'user', 'content': prompt})
        
        response = hf_client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=4096,
        )
        elapsed = time.time() - start
        content = response.choices[0].message.content
        usage = response.usage
        input_tokens = getattr(usage, 'prompt_tokens', 0) if usage else 0
        output_tokens = getattr(usage, 'completion_tokens', 0) if usage else 0
        total_tokens = (getattr(usage, 'total_tokens', 0) if usage else 0) or (input_tokens + output_tokens)
    
    else:
        raise ValueError(f'Unknown provider: {provider}')
    
    # Estimate cost
    cfg = MODEL_CONFIG[provider]
    cost = (input_tokens * cfg['cost_per_1m_input'] + output_tokens * cfg['cost_per_1m_output']) / 1_000_000
    
    meta = {
        'provider': provider,
        'model': model,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': total_tokens,
        'latency': elapsed,
        'cost_usd': cost,
    }
    
    return content, meta


def clean_json(content):
    """Clean LLM output to valid JSON."""
    content = content.strip()
    content = re.sub(r'^```json', '', content, flags=re.IGNORECASE).strip()
    content = re.sub(r'^```', '', content, flags=re.IGNORECASE).strip()
    content = re.sub(r'```$', '', content, flags=re.IGNORECASE).strip()
    idx = content.find('{')
    if idx > 0:
        content = content[idx:]
    idx2 = content.rfind('}')
    if idx2 > 0:
        content = content[:idx2+1]
    return content


def try_parse_json(content):
    """Try to parse JSON, return (parsed, success)."""
    try:
        cleaned = clean_json(content)
        return json.loads(cleaned), True
    except Exception:
        return {}, False


print('Unified LLM call function ready.')

In [ ]:
# === LOAD REAL SURVEY DATA ===
with open(PROJECT_ROOT / 'data/results/structured_summary_checkpoint.json', 'r') as f:
    checkpoint_data = json.load(f)

print(f'Loaded {len(checkpoint_data)} survey variables from checkpoint')

# Build identity-related survey results string (same as test_prompts.ipynb)
identity_vars = {k: v for k, v in checkpoint_data.items() 
                 if '|IDE' in k and k in list(checkpoint_data.keys())[:15]}

result_entries = []
for i, (var_id, data) in enumerate(identity_vars.items(), 1):
    result_entries.append(f'* QUESTION_ID: {var_id}')
    result_entries.append(f'  SUMMARY: {data["summary"]}')

survey_results_str = '\n'.join(result_entries)
print(f'Prepared {len(identity_vars)} survey variables ({len(survey_results_str)} chars)')

In [ ]:
# === RESULTS ACCUMULATOR ===
# Stores all test results for final comparison
all_results = defaultdict(lambda: defaultdict(dict))

def record_result(test_name, provider, meta, parsed=None, success=True, quality=None):
    """Store a test result for comparison."""
    all_results[test_name][provider] = {
        **meta,
        'success': success,
        'quality': quality,
        'parsed': parsed,
    }

def compare_table(test_name, metrics=None):
    """Display a comparison table for a test."""
    if metrics is None:
        metrics = ['total_tokens', 'latency', 'cost_usd', 'success']
    
    results = all_results[test_name]
    if not results:
        print(f'No results for {test_name}')
        return
    
    providers = list(results.keys())
    header = '| Metric | ' + ' | '.join(p.title() for p in providers) + ' |'
    sep = '|--------' + '|--------' * len(providers) + '|'
    rows = []
    for m in metrics:
        vals = []
        for p in providers:
            v = results[p].get(m, 'N/A')
            if isinstance(v, float):
                if m == 'cost_usd':
                    vals.append(f'${v:.6f}')
                elif m == 'latency':
                    vals.append(f'{v:.2f}s')
                else:
                    vals.append(f'{v:.2f}')
            else:
                vals.append(str(v))
        rows.append(f'| {m} | ' + ' | '.join(vals) + ' |')
    
    display(Markdown('\n'.join([header, sep] + rows)))

print('Results accumulator ready.')

---
## Test 1: Intent Classification

Classify user queries into one of 8 intents. Tests JSON compliance, intent accuracy, and confidence scoring.

In [ ]:
# === TEST 1: INTENT CLASSIFICATION ===

intent_dict = {
    'answer_general_questions': 'If the user asks questions about the project, the datasets, the methodology or the team.',
    'continue_conversation': 'If the user asks what you can do.',
    'query_variable_database': 'If the user asks a question that should be answered by querying survey variables.',
    'review_variable_selection': 'If the user asks to review the selection of variables.',
    'select_analysis_type': 'If the user asks for simple or complex analysis.',
    'confirm_and_run': 'If the user asks to produce an analysis or report.',
    'reset_conversation': 'If the user asks to start over.',
    'end_conversation': 'If the user asks to end the conversation.'
}
intent_str = '\n'.join([f'*{act}: {cond}' for act, cond in intent_dict.items()])

# Expected results for validation
test_queries_intent = [
    ('What do Mexicans think about education?', 'query_variable_database'),
    ('What topics are available?', 'answer_general_questions'),
    ('How do people feel about corruption?', 'query_variable_database'),
    ('What is this project about?', 'answer_general_questions'),
    ('Run the analysis', 'confirm_and_run'),
    ('I want a detailed analysis', 'select_analysis_type'),
]

def build_intent_prompt(user_message):
    return f"""
You are an intent classifier for a dataset analysis assistant.
Classify the user's intent into exactly one of the available intents.

Available intents:
{intent_str}

User message: "{user_message}"

CRITICAL DISTINCTION:
- Questions ABOUT the data/variables themselves ("What do people think about X?") -> query_variable_database
- Questions ABOUT what data/topics are AVAILABLE ("What topics do you have?") -> answer_general_questions

Respond with JSON: {{"intent": "...", "confidence": 0.0}}
"""

display(Markdown('## Test 1: Intent Classification'))
display(Markdown(f'Running {len(test_queries_intent)} queries across {len(ACTIVE_PROVIDERS)} providers...'))

intent_results = defaultdict(list)  # provider -> list of (correct, meta)

for provider in ACTIVE_PROVIDERS:
    display(Markdown(f'### {provider.title()}'))
    correct_count = 0
    total_tokens = 0
    total_latency = 0
    total_cost = 0
    
    for user_msg, expected_intent in test_queries_intent:
        prompt = build_intent_prompt(user_msg)
        content, meta = call_llm(prompt, provider=provider, temperature=0.0)
        parsed, success = try_parse_json(content)
        
        actual_intent = parsed.get('intent', 'PARSE_FAIL')
        is_correct = actual_intent == expected_intent
        if is_correct:
            correct_count += 1
        
        total_tokens += meta['total_tokens']
        total_latency += meta['latency']
        total_cost += meta['cost_usd']
        
        mark = 'OK' if is_correct else 'WRONG'
        print(f'  [{mark}] "{user_msg[:40]}..." -> {actual_intent} (expected: {expected_intent}) [{meta["latency"]:.2f}s]')
    
    accuracy = correct_count / len(test_queries_intent)
    record_result('intent_classification', provider, {
        'total_tokens': total_tokens,
        'latency': total_latency,
        'cost_usd': total_cost,
        'accuracy': accuracy,
    }, success=True, quality=accuracy * 100)
    print(f'  Accuracy: {accuracy:.0%} | Tokens: {total_tokens} | Latency: {total_latency:.2f}s | Cost: ${total_cost:.6f}')
    print()

display(Markdown('### Intent Classification Comparison'))
compare_table('intent_classification', ['total_tokens', 'latency', 'cost_usd', 'accuracy'])

---
## Test 2: Project Description

Answer general questions about the "Los Mexicanos" survey project. Tests factual accuracy and redirect detection.

In [ ]:
# === TEST 2: PROJECT DESCRIPTION ===

topics_list = [
    'Identidad Y Valores', 'Medio Ambiente', 'Ciencia Y Tecnologia',
    'Cultura', 'Derechos Humanos', 'Discriminacion',
    'Economia', 'Educacion', 'Familia', 'Justicia',
    'Migracion', 'Pobreza', 'Politica', 'Religion',
    'Salud', 'Seguridad', 'Sociedad'
]
topics_str = ', '.join(topics_list)

project_description = f"""
General information about the project:
-Name: "Los mexicanos vistos por si mismos"
-Description: This project is a group of {len(topics_list)} public opinion surveys conducted in Mexico between 2014 and 2015.
-Topics: {topics_str}.
-Objective: To understand the Mexican population's opinions on various topics.
-Team: Mtra. Julia Flores coordinated the team at the "Unidad de Investigacion en Opinion Publica".
-Repository: http://www.losmexicanos.unam.mx/
-Sponsor: Universidad Nacional Autonoma de Mexico (UNAM)
-Samples: All samples have size 1000, margin of error 3%, confidence 95%.
"""

project_queries = [
    ('What topics are covered in this survey project?', False),   # factual, no redirect
    ('Who funded this project?', False),                          # factual
    ('What do Mexicans think about healthcare?', True),           # should redirect
]

def build_project_prompt(user_query):
    return f"""
You are a helpful assistant answering general questions about a survey project.
Read the project description and reply with relevant information.
IMPORTANT: If the user asks about specific variables, redirect them to QUERY VARIABLE_DATABASE.

Project description:
{project_description}

User query: "{user_query}"

Respond with JSON: {{"answer": "...", "redirect_needed": false, "suggested_action": ""}}
"""

display(Markdown('## Test 2: Project Description'))

for provider in ACTIVE_PROVIDERS:
    display(Markdown(f'### {provider.title()}'))
    total_tokens = 0
    total_latency = 0
    total_cost = 0
    
    for user_query, expect_redirect in project_queries:
        prompt = build_project_prompt(user_query)
        content, meta = call_llm(prompt, provider=provider, temperature=0.3)
        parsed, success = try_parse_json(content)
        
        total_tokens += meta['total_tokens']
        total_latency += meta['latency']
        total_cost += meta['cost_usd']
        
        answer = parsed.get('answer', 'PARSE_FAIL')[:120]
        redirect = parsed.get('redirect_needed', False)
        print(f'  Q: "{user_query}"')
        print(f'  A: {answer}...')
        print(f'  Redirect: {redirect} (expected: {expect_redirect}) [{meta["latency"]:.2f}s]')
        print()
    
    record_result('project_description', provider, {
        'total_tokens': total_tokens,
        'latency': total_latency,
        'cost_usd': total_cost,
    })

display(Markdown('### Project Description Comparison'))
compare_table('project_description')

---
## Test 3: Variable Relevance Grading

Grade how relevant a survey variable is to a user query (0-3 scale). Tests scoring accuracy and JSON compliance.

In [ ]:
# === TEST 3: VARIABLE RELEVANCE GRADING ===

grading_query = "What do Mexicans think about national identity and pride?"
grading_vars = ['p5_1|IDE', 'p6|IDE', 'p3_5|IDE', 'p10_1|IDE', 'p9|IDE']

def build_grading_prompt(user_query, survey_info):
    return f"""
You are an expert in survey research, fluent in English and Spanish. Reply in English only.
Grade the relevance of the SURVEY INFORMATION to the user QUERY.

THE SURVEY INFORMATION has 3 parts:
- QUESTION: the question asked in the survey
- SUMMARY: a summary of the survey results
- IMPLICATIONS: expert analysis of the results

GRADE (0-3):
- 0: Completely unrelated
- 1: Some connection but mostly unrelated
- 2: Moderately relevant - moderately related, but adding important insights
- 3: Highly relevant - directly addresses the query

Return strict JSON: {{"GRADE": number, "EXPLANATION": "text"}}

QUERY: {user_query}
SURVEY_INFORMATION: {survey_info}
"""

display(Markdown('## Test 3: Variable Relevance Grading'))
display(Markdown(f'**Query:** "{grading_query}"'))

for provider in ACTIVE_PROVIDERS:
    display(Markdown(f'### {provider.title()}'))
    total_tokens = 0
    total_latency = 0
    total_cost = 0
    grades = []
    
    for var_id in grading_vars:
        if var_id not in checkpoint_data:
            continue
        data = checkpoint_data[var_id]
        info = f"SUMMARY: {data['summary']} IMPLICATIONS: {data['implication']}"
        prompt = build_grading_prompt(grading_query, info)
        
        content, meta = call_llm(prompt, provider=provider, temperature=0.0)
        parsed, success = try_parse_json(content)
        
        total_tokens += meta['total_tokens']
        total_latency += meta['latency']
        total_cost += meta['cost_usd']
        
        grade = parsed.get('GRADE', -1)
        grades.append(grade)
        expl = parsed.get('EXPLANATION', 'N/A')[:100]
        print(f'  {var_id}: Grade {grade}/3 - {expl}')
    
    avg_grade = sum(grades) / len(grades) if grades else 0
    record_result('variable_grading', provider, {
        'total_tokens': total_tokens,
        'latency': total_latency,
        'cost_usd': total_cost,
        'avg_grade': avg_grade,
        'grades': str(grades),
    })
    print(f'  Avg grade: {avg_grade:.1f} | Tokens: {total_tokens} | Cost: ${total_cost:.6f}')
    print()

display(Markdown('### Variable Grading Comparison'))
compare_table('variable_grading', ['total_tokens', 'latency', 'cost_usd', 'avg_grade', 'grades'])

---
## Test 4: Cross-Analysis Pattern Extraction

The core analysis prompt: find similar/different patterns across survey data. Tests all 3 prompt versions (V1, V2, V3) across providers.

In [ ]:
# === TEST 4: CROSS-ANALYSIS ===

from meta_prompting import PromptTemplates
from prompt_integration import PromptQualityAssessor

assessor = PromptQualityAssessor()

cross_query = "What do Mexicans think about their national identity?"
n_topics = 3
format_instr = 'Return strict JSON only. Keys: SIMILAR_PATTERN_1..N, DIFFERENT_PATTERN_1..N. Each has TITLE_SUMMARY, VARIABLE_STRING, DESCRIPTION.'

templates = {
    'V1': PromptTemplates.CROSS_ANALYSIS_V1,
    'V2': PromptTemplates.CROSS_ANALYSIS_V2,
    'V3': PromptTemplates.CROSS_ANALYSIS_V3,
}

display(Markdown('## Test 4: Cross-Analysis Pattern Extraction'))
display(Markdown(f'**Query:** "{cross_query}"  |  **Patterns:** {n_topics} similar + {n_topics} different'))

# We'll test V2 (the optimized version) across all providers for the main comparison,
# and also test all 3 versions with each provider.

cross_results = {}  # (provider, version) -> {quality, tokens, latency, patterns, cost}

for provider in ACTIVE_PROVIDERS:
    display(Markdown(f'### {provider.title()}'))
    
    for ver_name, template in templates.items():
        prompt = template.format(
            n_topics=n_topics,
            user_query=cross_query,
            results=survey_results_str,
            format_instructions=format_instr
        )
        
        content, meta = call_llm(prompt, provider=provider, temperature=0.5)
        parsed, success = try_parse_json(content)
        
        if success:
            quality = assessor.assess_cross_analysis_quality(content, parsed, n_topics * 2)
            patterns = len([k for k in parsed if 'PATTERN' in k])
        else:
            quality = 0
            patterns = 0
        
        cross_results[(provider, ver_name)] = {
            'quality': quality,
            'tokens': meta['total_tokens'],
            'latency': meta['latency'],
            'cost': meta['cost_usd'],
            'patterns': patterns,
            'json_valid': success,
        }
        
        print(f'  {ver_name}: Quality={quality:.0f}/100 | Patterns={patterns}/{n_topics*2} | '
              f'Tokens={meta["total_tokens"]} | {meta["latency"]:.2f}s | ${meta["cost_usd"]:.6f} | JSON={success}')
    
    # Record V2 result for main comparison
    v2 = cross_results[(provider, 'V2')]
    record_result('cross_analysis', provider, {
        'total_tokens': v2['tokens'],
        'latency': v2['latency'],
        'cost_usd': v2['cost'],
        'quality': v2['quality'],
        'patterns_found': v2['patterns'],
        'json_valid': v2['json_valid'],
    }, quality=v2['quality'])
    print()

In [ ]:
# === CROSS-ANALYSIS: DETAILED COMPARISON TABLE ===

display(Markdown('### Cross-Analysis: Full Comparison (V2 across providers)'))
compare_table('cross_analysis', ['total_tokens', 'latency', 'cost_usd', 'quality', 'patterns_found', 'json_valid'])

# Full version x provider matrix
display(Markdown('### Cross-Analysis: Version x Provider Matrix'))

# Build table
header_cols = []
for p in ACTIVE_PROVIDERS:
    header_cols.extend([f'{p.title()} Quality', f'{p.title()} Tokens', f'{p.title()} Cost'])

header = '| Version | ' + ' | '.join(header_cols) + ' |'
sep = '|---------|' + '|--------' * len(header_cols) + '|'
rows = []
for ver in ['V1', 'V2', 'V3']:
    vals = []
    for p in ACTIVE_PROVIDERS:
        r = cross_results.get((p, ver), {})
        vals.append(f'{r.get("quality", 0):.0f}/100')
        vals.append(str(r.get('tokens', 'N/A')))
        vals.append(f'${r.get("cost", 0):.6f}')
    rows.append(f'| {ver} | ' + ' | '.join(vals) + ' |')

display(Markdown('\n'.join([header, sep] + rows)))

---
## Test 5: Expert Summary

Given a pattern from cross-analysis, generate expert-level interpretation.

In [ ]:
# === TEST 5: EXPERT SUMMARY ===

# Use a fixed pattern description for consistency across providers
expert_pattern = {
    'title': 'Strong National Pride with Cultural Ambivalence',
    'variables': 'p5_1|IDE, p6|IDE, p3_5|IDE',
    'description': 'Mexicans express strong pride in national identity (85% proud, p5_1|IDE) '
                   'but show ambivalence toward cultural traditions (p3_5|IDE), with younger '
                   'generations showing lower attachment to traditional values (p6|IDE).'
}

# Build expert implications
imp_vars = ['p5_1|IDE', 'p6|IDE', 'p3_5|IDE']
implications = [checkpoint_data[v]['implication'] for v in imp_vars if v in checkpoint_data]
imp_str = ' * '.join(implications) if implications else 'No expert implications available.'

def build_expert_prompt(pattern, imp_str):
    return f"""
You are a very thorough research assistant working on a survey research project.
The objective is to study public opinion on various topics.
You are fully bilingual in English and Spanish. Reply in English.

Read the EXPERT STATEMENTS from experts about what information they consider important.
Then read the SURVEY SUMMARY and write a one-paragraph reply elaborating on how the results relate to their concerns.
Include as many relevant facts (numbers) as possible, citing QUESTION_IDs in parentheses (e.g., p5_1|IDE).

EXPERT STATEMENTS: * {imp_str}
SURVEY SUMMARY: {pattern['description']}

Return JSON: {{"EXPERT_REPLY": "your paragraph here"}}
"""

display(Markdown('## Test 5: Expert Summary'))
display(Markdown(f'**Pattern:** "{expert_pattern["title"]}"'))

for provider in ACTIVE_PROVIDERS:
    display(Markdown(f'### {provider.title()}'))
    prompt = build_expert_prompt(expert_pattern, imp_str)
    content, meta = call_llm(prompt, provider=provider, temperature=0.7)
    parsed, success = try_parse_json(content)
    
    quality = assessor.assess_expert_summary_quality(content, parsed) if success else 0
    reply = parsed.get('EXPERT_REPLY', 'PARSE_FAIL')[:400]
    
    print(f'  Tokens: {meta["total_tokens"]} | Latency: {meta["latency"]:.2f}s | Cost: ${meta["cost_usd"]:.6f}')
    print(f'  Quality: {quality:.0f}/100 | JSON valid: {success}')
    display(Markdown(f'> {reply}'))
    
    record_result('expert_summary', provider, {
        'total_tokens': meta['total_tokens'],
        'latency': meta['latency'],
        'cost_usd': meta['cost_usd'],
        'quality': quality,
        'json_valid': success,
    }, quality=quality)
    print()

display(Markdown('### Expert Summary Comparison'))
compare_table('expert_summary', ['total_tokens', 'latency', 'cost_usd', 'quality', 'json_valid'])

---
## Test 6: Transversal Analysis

Synthesize expert summaries into topic summaries, a general overview, and a direct query answer.

In [ ]:
# === TEST 6: TRANSVERSAL ANALYSIS ===

# Fixed expert statements for consistency
expert_statements = [
    'Mexicans show strong national pride (85% proud, p5_1|IDE) but this pride is complex and multifaceted.',
    'Cultural traditions face generational tension: older Mexicans value customs more (72%, p3_5|IDE) vs younger groups (54%).',
    'National symbols evoke strong emotional responses (p6|IDE), serving as unifying factors across socioeconomic divides.',
    'Regional identity competes with national identity, particularly in northern and southern border states (p9|IDE).'
]
all_statements = ' * '.join(expert_statements)

trans_query = "What do Mexicans think about their national identity?"

def build_transversal_prompt(query, statements):
    return f"""
You are a thorough research assistant and expert in survey research and public opinion.
You are fully bilingual in English and Spanish.

Perform three analyses and return a single JSON dictionary:

1) Read the STATEMENTS from experts. Identify 3 common topics across them.
   Write a one-paragraph summary of each topic, citing numbers and QUESTION_IDs.
   Save as TOPIC_SUMMARIES (dict with topic names as ALL CAPS keys).

2) Write a one-paragraph summary of the most relevant results for a general audience.
   Include facts and QUESTION_IDs. Save as TOPIC_SUMMARY.

3) Write a two-sentence answer to the QUERY summarizing the most important points.
   Do not include numbers, just ideas. Save as QUERY_ANSWER.

STATEMENTS: {statements}

QUERY: {query}

Reply in the language of the QUERY. Return only valid JSON.
Return JSON with keys: TOPIC_SUMMARIES, TOPIC_SUMMARY, QUERY_ANSWER
"""

display(Markdown('## Test 6: Transversal Analysis'))

for provider in ACTIVE_PROVIDERS:
    display(Markdown(f'### {provider.title()}'))
    prompt = build_transversal_prompt(trans_query, all_statements)
    content, meta = call_llm(prompt, provider=provider, temperature=0.7)
    parsed, success = try_parse_json(content)
    
    has_all_keys = all(k in parsed for k in ['TOPIC_SUMMARIES', 'TOPIC_SUMMARY', 'QUERY_ANSWER']) if success else False
    n_topics_found = len(parsed.get('TOPIC_SUMMARIES', {})) if success else 0
    
    print(f'  Tokens: {meta["total_tokens"]} | Latency: {meta["latency"]:.2f}s | Cost: ${meta["cost_usd"]:.6f}')
    print(f'  JSON valid: {success} | All keys present: {has_all_keys} | Topics found: {n_topics_found}')
    
    if success and 'QUERY_ANSWER' in parsed:
        display(Markdown(f'**Direct Answer:** {parsed["QUERY_ANSWER"]}'))
    
    record_result('transversal', provider, {
        'total_tokens': meta['total_tokens'],
        'latency': meta['latency'],
        'cost_usd': meta['cost_usd'],
        'json_valid': success,
        'all_keys': has_all_keys,
        'topics_found': n_topics_found,
    })
    print()

display(Markdown('### Transversal Analysis Comparison'))
compare_table('transversal', ['total_tokens', 'latency', 'cost_usd', 'json_valid', 'all_keys', 'topics_found'])

---
## Test 7: Full Pipeline (End-to-End)

Run the complete chain: Intent -> Grading -> Cross-Analysis -> Expert Summary -> Transversal. Measure total cost and latency.

In [ ]:
# === TEST 7: FULL PIPELINE ===

PIPELINE_QUERY = "What do Mexicans think about cultural diversity and preserving traditions?"

display(Markdown(f'## Test 7: Full Pipeline'))
display(Markdown(f'**Query:** "{PIPELINE_QUERY}"'))

pipeline_vars = ['p8|IDE', 'p9|IDE', 'p4|IDE', 'p5_1|IDE', 'p10_1|IDE']

for provider in ACTIVE_PROVIDERS:
    display(Markdown(f'### {provider.title()} - Full Pipeline'))
    pipe_start = time.time()
    pipe_tokens = 0
    pipe_cost = 0
    pipe_steps = []
    
    # Step 1: Intent
    prompt = build_intent_prompt(PIPELINE_QUERY)
    content, meta = call_llm(prompt, provider=provider, temperature=0.0)
    parsed, _ = try_parse_json(content)
    pipe_tokens += meta['total_tokens']
    pipe_cost += meta['cost_usd']
    intent = parsed.get('intent', 'unknown')
    pipe_steps.append(f'Intent: {intent} ({meta["latency"]:.2f}s)')
    
    # Step 2: Grade 5 variables
    grading_results = {}
    for var_id in pipeline_vars:
        if var_id not in checkpoint_data:
            continue
        data = checkpoint_data[var_id]
        info = f"SUMMARY: {data['summary']} IMPLICATIONS: {data['implication']}"
        prompt = build_grading_prompt(PIPELINE_QUERY, info)
        content, meta = call_llm(prompt, provider=provider, temperature=0.0)
        parsed, _ = try_parse_json(content)
        pipe_tokens += meta['total_tokens']
        pipe_cost += meta['cost_usd']
        grading_results[var_id] = parsed.get('GRADE', 0)
    relevant = [k for k, v in grading_results.items() if v >= 2]
    pipe_steps.append(f'Grading: {len(relevant)}/{len(grading_results)} relevant')
    
    # Step 3: Cross-analysis (V2)
    prompt = PromptTemplates.CROSS_ANALYSIS_V2.format(
        n_topics=2,
        user_query=PIPELINE_QUERY,
        results=survey_results_str,
        format_instructions=format_instr
    )
    content, meta = call_llm(prompt, provider=provider, temperature=0.5)
    parsed_cross, cross_success = try_parse_json(content)
    pipe_tokens += meta['total_tokens']
    pipe_cost += meta['cost_usd']
    patterns_found = len([k for k in parsed_cross if 'PATTERN' in k]) if cross_success else 0
    pipe_steps.append(f'Cross-analysis: {patterns_found} patterns ({meta["latency"]:.2f}s)')
    
    # Step 4: Expert summaries for each pattern
    expert_replies = []
    if cross_success:
        for key, val in parsed_cross.items():
            if 'PATTERN' not in key or not isinstance(val, dict):
                continue
            var_ids = [v.strip() for v in val.get('VARIABLE_STRING', '').split(',')]
            imps = [checkpoint_data[v]['implication'] for v in var_ids if v in checkpoint_data]
            imp_text = ' * '.join(imps) if imps else 'No expert implications available.'
            
            ep = build_expert_prompt({'description': val.get('DESCRIPTION', '')}, imp_text)
            ec, em = call_llm(ep, provider=provider, temperature=0.7)
            er, _ = try_parse_json(ec)
            expert_replies.append(er.get('EXPERT_REPLY', ''))
            pipe_tokens += em['total_tokens']
            pipe_cost += em['cost_usd']
    pipe_steps.append(f'Expert summaries: {len(expert_replies)}')
    
    # Step 5: Transversal
    all_expert = ' * '.join(expert_replies) if expert_replies else 'No expert summaries available.'
    prompt = build_transversal_prompt(PIPELINE_QUERY, all_expert)
    content, meta = call_llm(prompt, provider=provider, temperature=0.7)
    parsed_trans, trans_success = try_parse_json(content)
    pipe_tokens += meta['total_tokens']
    pipe_cost += meta['cost_usd']
    
    pipe_elapsed = time.time() - pipe_start
    
    # Report
    for step in pipe_steps:
        print(f'  {step}')
    print(f'  Transversal: {"OK" if trans_success else "FAIL"} ({meta["latency"]:.2f}s)')
    print(f'  ---')
    print(f'  TOTAL: {pipe_tokens} tokens | {pipe_elapsed:.1f}s | ${pipe_cost:.6f}')
    
    if trans_success and 'QUERY_ANSWER' in parsed_trans:
        display(Markdown(f'**Answer:** {parsed_trans["QUERY_ANSWER"]}'))
    
    record_result('full_pipeline', provider, {
        'total_tokens': pipe_tokens,
        'latency': pipe_elapsed,
        'cost_usd': pipe_cost,
        'pipeline_success': trans_success,
        'patterns_found': patterns_found,
    })
    print()

display(Markdown('### Full Pipeline Comparison'))
compare_table('full_pipeline', ['total_tokens', 'latency', 'cost_usd', 'pipeline_success', 'patterns_found'])

---
## Final Summary: Provider Comparison Dashboard

In [ ]:
# === FINAL SUMMARY ===

display(Markdown('# Final Multi-Model Comparison Summary'))

# Aggregate all results
summary = defaultdict(lambda: {'total_tokens': 0, 'total_latency': 0, 'total_cost': 0, 'tests_passed': 0, 'tests_total': 0})

for test_name, providers in all_results.items():
    for provider, data in providers.items():
        summary[provider]['total_tokens'] += data.get('total_tokens', 0)
        summary[provider]['total_latency'] += data.get('latency', 0)
        summary[provider]['total_cost'] += data.get('cost_usd', 0)
        summary[provider]['tests_total'] += 1
        if data.get('success', True):
            summary[provider]['tests_passed'] += 1

# Display summary table
header = '| Metric | ' + ' | '.join(p.title() for p in ACTIVE_PROVIDERS) + ' |'
sep = '|--------' + '|--------' * len(ACTIVE_PROVIDERS) + '|'

rows = []
metrics_display = [
    ('Total Tokens (all tests)', 'total_tokens', lambda x: f'{x:,}'),
    ('Total Latency', 'total_latency', lambda x: f'{x:.1f}s'),
    ('Total Cost', 'total_cost', lambda x: f'${x:.6f}'),
    ('Tests Passed', 'tests_passed', lambda x: str(x)),
]

for label, key, fmt in metrics_display:
    vals = [fmt(summary[p][key]) for p in ACTIVE_PROVIDERS]
    rows.append(f'| {label} | ' + ' | '.join(vals) + ' |')

# Cost per 1000 calls estimate
cost_row_vals = []
for p in ACTIVE_PROVIDERS:
    s = summary[p]
    if s['tests_total'] > 0:
        avg_cost = s['total_cost'] / s['tests_total']
        cost_1k = avg_cost * 1000
        cost_row_vals.append(f'${cost_1k:.2f}')
    else:
        cost_row_vals.append('N/A')
rows.append(f'| Est. Cost/1000 calls | ' + ' | '.join(cost_row_vals) + ' |')

display(Markdown('\n'.join([header, sep] + rows)))

# Per-test breakdown
display(Markdown('### Per-Test Cost Breakdown'))
for test_name in all_results:
    display(Markdown(f'**{test_name}**'))
    compare_table(test_name, ['total_tokens', 'latency', 'cost_usd'])

In [ ]:
# === VISUALIZATION: COST vs LATENCY vs QUALITY ===
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

test_names = list(all_results.keys())
colors = {'openai': '#74aa9c', 'anthropic': '#d4a574', 'gemini': '#7494aa', 'huggingface': '#ff9d00'}

# Chart 1: Total tokens per test
ax = axes[0]
x = np.arange(len(test_names))
width = 0.8 / len(ACTIVE_PROVIDERS)
for i, provider in enumerate(ACTIVE_PROVIDERS):
    vals = [all_results[t].get(provider, {}).get('total_tokens', 0) for t in test_names]
    ax.bar(x + i * width, vals, width, label=provider.title(), color=colors.get(provider, '#999'))
ax.set_xlabel('Test')
ax.set_ylabel('Total Tokens')
ax.set_title('Token Usage by Test')
ax.set_xticks(x + width * (len(ACTIVE_PROVIDERS) - 1) / 2)
ax.set_xticklabels([t.replace('_', '\n') for t in test_names], fontsize=8)
ax.legend()

# Chart 2: Latency per test
ax = axes[1]
for i, provider in enumerate(ACTIVE_PROVIDERS):
    vals = [all_results[t].get(provider, {}).get('latency', 0) for t in test_names]
    ax.bar(x + i * width, vals, width, label=provider.title(), color=colors.get(provider, '#999'))
ax.set_xlabel('Test')
ax.set_ylabel('Latency (seconds)')
ax.set_title('Latency by Test')
ax.set_xticks(x + width * (len(ACTIVE_PROVIDERS) - 1) / 2)
ax.set_xticklabels([t.replace('_', '\n') for t in test_names], fontsize=8)
ax.legend()

# Chart 3: Cost per test
ax = axes[2]
for i, provider in enumerate(ACTIVE_PROVIDERS):
    vals = [all_results[t].get(provider, {}).get('cost_usd', 0) * 1000 for t in test_names]
    ax.bar(x + i * width, vals, width, label=provider.title(), color=colors.get(provider, '#999'))
ax.set_xlabel('Test')
ax.set_ylabel('Cost ($ per 1000 calls)')
ax.set_title('Estimated Cost per 1000 Calls')
ax.set_xticks(x + width * (len(ACTIVE_PROVIDERS) - 1) / 2)
ax.set_xticklabels([t.replace('_', '\n') for t in test_names], fontsize=8)
ax.legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data/results/multi_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to data/results/multi_model_comparison.png')

In [ ]:
# === SAVE RAW RESULTS TO JSON ===

# Convert defaultdicts to regular dicts for serialization
export_results = {}
for test_name, providers in all_results.items():
    export_results[test_name] = {}
    for provider, data in providers.items():
        # Remove non-serializable items
        export_data = {k: v for k, v in data.items() if k != 'parsed'}
        export_results[test_name][provider] = export_data

output_path = PROJECT_ROOT / 'data/results/multi_model_comparison.json'
with open(output_path, 'w') as f:
    json.dump(export_results, f, indent=2, default=str)

print(f'Results saved to {output_path}')
print(f'Tests run: {len(export_results)}')
print(f'Providers: {ACTIVE_PROVIDERS}')

---
## Conclusions

### Key Dimensions to Compare

| Dimension | What to Look For |
|-----------|------------------|
| **JSON Compliance** | Which provider returns valid JSON most reliably? |
| **Quality Score** | Which produces the best-structured cross-analysis patterns? |
| **Latency** | Which is fastest for interactive use (dashboard)? |
| **Cost** | Which is most economical for batch processing (1000s of variables)? |
| **Spanish/Bilingual** | Which handles bilingual content (Spanish survey data, English output) best? |
| **Instruction Following** | Which follows complex multi-step prompts most faithfully? |

### Recommendations

After running all tests, review the summary table above and consider:
1. **If cost is primary concern**: Check the cost column - Gemini Flash is typically cheapest
2. **If quality is primary**: Compare quality scores on cross-analysis and expert summary
3. **If latency matters**: Dashboard use needs sub-2s responses for good UX
4. **For production**: Consider a mixed strategy (e.g., cheap model for intent, premium for analysis)